# Updating an old model with new features
Old X patients (age, sex, complaint) → base model. Later, X new patients with vitals V1–V3. Three ways to add the vitals, compared on the same test set.

## 1. Data
Old batch, new batch and test set, all from the same simulated population.

### 1.1 Setup

In [1]:
import os

# must be set before pymc/jax are imported
os.environ["JAX_PLATFORMS"] = "cpu"                                     # GPU was slower for this model
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=6"    # one JAX device per chain, run in parallel

import numpy as np
import pandas as pd
import pymc as pm

SEED = 42

### 1.2 Patients

In [2]:
rng = np.random.default_rng(SEED)
N_OLD, N_NEW, N_TEST = 20_000, 20_000, 20_000
N = N_OLD + N_NEW + N_TEST
COMPLAINTS = ["chest pain", "shortness of breath", "headache", "abdominal pain"]

df = pd.DataFrame({
    "age": rng.normal(55, 18, N).clip(18, 100).round(),
    "male": rng.binomial(1, 0.5, N),
    "complaint": rng.choice(COMPLAINTS, N),
})
# vital signs, shaped after the real ED patients in data.csv.xls
# the old patients have vitals too (they drive their risk), but no model is ever shown them
fever = rng.random(N) < 0.10                      # 10% febrile
cold = rng.random(N) < 0.005                      # 0.5% hypothermic
V1 = rng.normal(36.55, 0.4, N) + fever * rng.gamma(2.0, 0.6, N)
df["V1"] = np.where(cold, rng.normal(34.6, 0.6, N), V1).clip(30, 42.5).round(1)                   # body temperature, °C
df["V2"] = (rng.lognormal(np.log(131), 0.2, N) + 0.4 * (df["age"] - 55)).clip(50, 260).round()     # systolic BP, mmHg
df["V3"] = (rng.lognormal(np.log(82), 0.21, N) + 10 * (df["V1"] - 36.6)).clip(30, 220).round()    # heart rate, bpm

# old: the first records, used once for the base model and then gone; new: arrive later with vitals; test: held out
df["group"] = np.repeat(["old", "new", "test"], [N_OLD, N_NEW, N_TEST])
in_old, in_new, in_test = (df["group"] == g for g in ["old", "new", "test"])
df.head()

,age,male,complaint,V1,V2,V3,group
0,60.0,1,chest pain,36.5,115.0,88.0,old
1,36.0,0,abdominal pain,36.1,118.0,87.0,old
2,69.0,1,shortness of breath,36.0,120.0,58.0,old
3,72.0,1,abdominal pain,36.7,166.0,71.0,old
4,20.0,1,headache,36.1,112.0,70.0,old


### 1.3 Features

In [3]:
# age is scaled with the old batch, as the base model was built on it; vitals with the new batch, where they first appear
AGE_MEAN, AGE_SD = df.loc[in_old, "age"].mean(), df.loc[in_old, "age"].std()
VITALS = ["V1", "V2", "V3"]
VITAL_MEAN, VITAL_SD = df.loc[in_new, VITALS].mean(), df.loc[in_new, VITALS].std()

X = pd.DataFrame({
    "age": (df["age"] - AGE_MEAN) / AGE_SD,
    "male": df["male"],
})
for c in COMPLAINTS[1:]:
    X[c] = (df["complaint"] == c).astype(int)
# each vital: a standardised linear term plus its square, so both low and high values can raise risk
for v in VITALS:
    z = (df[v] - VITAL_MEAN[v]) / VITAL_SD[v]
    X[v] = z
    X[v + "_sq"] = z ** 2

BASE_COLS = ["age", "male"] + COMPLAINTS[1:]
VITAL_COLS = [c for v in VITALS for c in (v, v + "_sq")]
ALL_COLS = BASE_COLS + VITAL_COLS
X.head()

,age,male,shortness of breath,headache,abdominal pain,V1,V1_sq,V2,V2_sq,V3,V3_sq
0,0.272484,1,0,0,0,-0.252569,0.063791,-0.670029,0.448939,0.190036,0.036114
1,-1.086172,0,0,0,1,-0.903645,0.816575,-0.563192,0.317186,0.137386,0.018875
2,0.781980,1,1,0,0,-1.066414,1.137239,-0.491968,0.242032,-1.389471,1.930630
3,0.951812,1,0,0,1,0.072969,0.005325,1.146198,1.313769,-0.705018,0.497051
4,-1.991943,1,0,1,0,-0.903645,0.816575,-0.776866,0.603521,-0.757668,0.574061


### 1.4 True model and deaths

In [4]:
# the true values, used to simulate deaths and to score the models
rng = np.random.default_rng(SEED)
TRUE_INTERCEPT = -4.2           # keeps mortality near 3% with the vitals added
TRUE_BETA = pd.Series({
    "age": 0.8,                   # per standard deviation of age (about 18 years)
    "male": 0.3,
    "shortness of breath": 0.5,
    "headache": -0.7,
    "abdominal pain": -0.2,
    # vitals, per standard deviation (linear) and per squared standard deviation (curvature)
    "V1": 0.1,   "V1_sq": 0.04,     # temperature: long fever tail, so a small square term
    "V2": -0.3,  "V2_sq": 0.1,      # systolic BP: low is worse than high
    "V3": 0.2,   "V3_sq": 0.1,      # heart rate
})

p_true = 1 / (1 + np.exp(-(TRUE_INTERCEPT + X[ALL_COLS] @ TRUE_BETA)))
df["dead"] = rng.binomial(1, p_true)
df.groupby("group", sort=False)["dead"].agg(patients="size", deaths="sum", rate="mean")

,patients,deaths,rate
group,,,
old,20000,626,0.03130
new,20000,634,0.03170
test,20000,569,0.02845


## 2. Base model
Fitted once on the old X K: age, sex and complaint, N(0, 1) priors.

### 2.1 Fit
$x_i$ = base features (age, sex, complaint), $y_i$ = 1 if patient $i$ died.

$$\operatorname{logit} p_i = \alpha + x_i^\top \beta, \qquad y_i \sim \text{Bernoulli}(p_i), \qquad \alpha,\ \beta_j \sim \mathcal{N}(0, 1)$$

$$p(\alpha, \beta \mid D_\text{old}) \;\propto\; \prod_{i \in \text{old}} p_i^{y_i} (1 - p_i)^{1 - y_i} \;\cdot\; \mathcal{N}(\alpha; 0, 1) \prod_j \mathcal{N}(\beta_j; 0, 1)$$

In [5]:
with pm.Model(coords={"feature": BASE_COLS}) as model_base:
    intercept = pm.Normal("intercept", mu=0, sigma=1)
    beta = pm.Normal("beta", mu=0, sigma=1, dims="feature")

    pm.Bernoulli("dead", logit_p=intercept + pm.math.dot(X.loc[in_old, BASE_COLS].values, beta),
                 observed=df.loc[in_old, "dead"].values)

    # 6 chains x 667 draws ≈ 4000 draws, sampled with NumPyro (JAX) on the CPU
    idata_base = pm.sample(667, tune=1000, chains=6, nuts_sampler="numpyro", random_seed=SEED)

NUTS[numpyro]: [intercept, beta]


/home/theo/Work/DEV/skola/pda/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  0%|          | 0/1667 [00:00<?, ?it/s]

Compiling.. :   0%|          | 0/1667 [00:00<?, ?it/s]

  0%|          | 0/1667 [00:00<?, ?it/s]

...

Running chain 0: 100%|██████████| 1667/1667 [00:13<00:00, 123.37it/s]


Running chain 3: 100%|██████████| 1667/1667 [00:13<00:00, 123.41it/s]


Running chain 5: 100%|██████████| 1667/1667 [00:13<00:00, 123.30it/s]

Running chain 4: 100%|█████████▉| 1660/1667 [00:13<00:00, 149.10it/s]

Running chain 4: 100%|██████████| 1667/1667 [00:13<00:00, 119.82it/s]

### 2.2 Base score
The base model as one fixed number per patient, using the posterior means $\bar\alpha, \bar\beta$:

$$s_i = \bar\alpha + x_i^\top \bar\beta$$

In [6]:
def draws_of(idata, cols, **fixed):
    """Posterior draws as a table: an intercept column and one per feature; `fixed` adds coefficients that are not sampled."""
    post = idata.posterior
    d = pd.DataFrame(post["beta"].values.reshape(-1, len(cols)), columns=cols)
    d.insert(0, "intercept", post["intercept"].values.ravel() if "intercept" in post else 0.0)
    return d.assign(**fixed)

base_draws = draws_of(idata_base, BASE_COLS)
# the base model as one fixed number per patient: its log-odds of death with the posterior-mean coefficients
X["base_lp"] = base_draws["intercept"].mean() + X[BASE_COLS] @ base_draws[BASE_COLS].mean()

pd.DataFrame({"true": pd.Series({"intercept": TRUE_INTERCEPT, **TRUE_BETA[BASE_COLS]}),
              "base mean": base_draws.mean(), "base sd": base_draws.std()}).round(2)

,true,base mean,base sd
intercept,-4.2,-3.68,0.10
age,0.8,0.73,0.04
male,0.3,0.28,0.08
shortness of breath,0.5,0.41,0.10
headache,-0.7,-1.06,0.15
abdominal pain,-0.2,-0.43,0.12


## 3. Adding the vitals
Each method is fitted independently on the new 10K. The old patients are gone; only the base model is left.

### 3.1 Fixed base + vitals
Base score as a fixed offset; only the vital coefficients are learned. $v_i$ = vital features ($V1, V1^2, \dots, V3^2$).

$$\operatorname{logit} p_i = s_i + v_i^\top \gamma, \qquad \gamma_k \sim \mathcal{N}(0, 1)$$

$$p(\gamma \mid D_\text{new}) \;\propto\; \prod_{i \in \text{new}} p_i^{y_i} (1 - p_i)^{1 - y_i} \;\cdot\; \prod_k \mathcal{N}(\gamma_k; 0, 1)$$

In [7]:
y_new = df.loc[in_new, "dead"].values

with pm.Model(coords={"feature": VITAL_COLS}) as model_fixed:
    beta = pm.Normal("beta", mu=0, sigma=1, dims="feature")

    pm.Bernoulli("dead", logit_p=X.loc[in_new, "base_lp"].values + pm.math.dot(X.loc[in_new, VITAL_COLS].values, beta),
                 observed=y_new)

    idata_fixed = pm.sample(667, tune=1000, chains=6, nuts_sampler="numpyro", random_seed=SEED)

NUTS[numpyro]: [beta]


  0%|          | 0/1667 [00:00<?, ?it/s]

Compiling.. :   0%|          | 0/1667 [00:00<?, ?it/s]

  0%|          | 0/1667 [00:00<?, ?it/s]

Compiling.. :   0%|          | 0/1667 [00:00<?, ?it/s]

...

Running chain 3:  95%|█████████▍| 1577/1667 [00:13<00:00, 129.37it/s]

Running chain 4: 100%|█████████▉| 1660/1667 [00:13<00:00, 131.85it/s]

Running chain 4: 100%|██████████| 1667/1667 [00:13<00:00, 121.18it/s]

Running chain 3: 100%|█████████▉| 1660/1667 [00:13<00:00, 138.62it/s]

Running chain 3: 100%|██████████| 1667/1667 [00:13<00:00, 120.21it/s]

### 3.2 Recalibrated base + vitals
New intercept and slope for the base score, plus the vital coefficients.

$$\operatorname{logit} p_i = a + b\, s_i + v_i^\top \gamma, \qquad a \sim \mathcal{N}(0, 1),\quad b \sim \mathcal{N}(1, 1),\quad \gamma_k \sim \mathcal{N}(0, 1)$$

$$p(a, b, \gamma \mid D_\text{new}) \;\propto\; \prod_{i \in \text{new}} p_i^{y_i} (1 - p_i)^{1 - y_i} \;\cdot\; \mathcal{N}(a; 0, 1)\, \mathcal{N}(b; 1, 1) \prod_k \mathcal{N}(\gamma_k; 0, 1)$$

In [8]:
RECAL_COLS = ["base_lp"] + VITAL_COLS

with pm.Model(coords={"feature": RECAL_COLS}) as model_recal:
    intercept = pm.Normal("intercept", mu=0, sigma=1)
    # the base score's slope is centred on 1 (keep it as is); the vitals start at 0
    beta = pm.Normal("beta", mu=np.r_[1.0, np.zeros(len(VITAL_COLS))], sigma=1, dims="feature")

    pm.Bernoulli("dead", logit_p=intercept + pm.math.dot(X.loc[in_new, RECAL_COLS].values, beta),
                 observed=y_new)

    idata_recal = pm.sample(667, tune=1000, chains=6, nuts_sampler="numpyro", random_seed=SEED)

NUTS[numpyro]: [intercept, beta]


  0%|          | 0/1667 [00:00<?, ?it/s]

Compiling.. :   0%|          | 0/1667 [00:00<?, ?it/s]

  0%|          | 0/1667 [00:00<?, ?it/s]

Compiling.. :   0%|          | 0/1667 [00:00<?, ?it/s]

...

Running chain 1: 100%|██████████| 1667/1667 [00:30<00:00, 54.13it/s]

Running chain 4: 100%|█████████▉| 1660/1667 [00:30<00:00, 55.33it/s]

Running chain 4: 100%|██████████| 1667/1667 [00:30<00:00, 53.95it/s]

Running chain 5: 100%|█████████▉| 1660/1667 [00:31<00:00, 59.98it/s]

Running chain 5: 100%|██████████| 1667/1667 [00:31<00:00, 53.65it/s]

### 3.3 Old posterior as prior
Full model; the base posterior is the prior for the shared parameters, N(0, 1) for the vitals.

$$\operatorname{logit} p_i = \alpha + x_i^\top \beta + v_i^\top \gamma, \qquad (\alpha, \beta) \sim \mathcal{N}(\mu_\text{old}, \Sigma_\text{old}), \qquad \gamma_k \sim \mathcal{N}(0, 1)$$

$\mu_\text{old}, \Sigma_\text{old}$ = mean and covariance of the base posterior draws, so $\mathcal{N}(\mu_\text{old}, \Sigma_\text{old}) \approx p(\alpha, \beta \mid D_\text{old})$:

$$p(\alpha, \beta, \gamma \mid D_\text{old}, D_\text{new}) \;\propto\; \prod_{i \in \text{new}} p_i^{y_i} (1 - p_i)^{1 - y_i} \;\cdot\; \underbrace{\mathcal{N}(\alpha, \beta;\, \mu_\text{old}, \Sigma_\text{old})}_{\approx\, p(\alpha, \beta \mid D_\text{old})} \prod_k \mathcal{N}(\gamma_k; 0, 1)$$

In [9]:
# prior for the parameters the base model has: mean and covariance of its posterior draws
prev = base_draws.values                                   # intercept + BASE_COLS

with pm.Model(coords={"feature": ALL_COLS}) as model_prior:
    old = pm.MvNormal("old", mu=prev.mean(axis=0), cov=np.cov(prev, rowvar=False))
    added = pm.Normal("added", mu=0, sigma=1, shape=len(VITAL_COLS))
    intercept = pm.Deterministic("intercept", old[0])
    beta = pm.Deterministic("beta", pm.math.concatenate([old[1:], added]), dims="feature")

    pm.Bernoulli("dead", logit_p=intercept + pm.math.dot(X.loc[in_new, ALL_COLS].values, beta),
                 observed=y_new)

    idata_prior = pm.sample(667, tune=1000, chains=6, nuts_sampler="numpyro", random_seed=SEED)

NUTS[numpyro]: [old, added]


  0%|          | 0/1667 [00:00<?, ?it/s]

Compiling.. :   0%|          | 0/1667 [00:00<?, ?it/s]

  0%|          | 0/1667 [00:00<?, ?it/s]

Compiling.. :   0%|          | 0/1667 [00:00<?, ?it/s]

...

Running chain 0: 100%|██████████| 1667/1667 [00:23<00:00, 72.19it/s]

Running chain 5: 100%|█████████▉| 1660/1667 [00:23<00:00, 90.54it/s]

Running chain 5: 100%|██████████| 1667/1667 [00:23<00:00, 71.36it/s]

Running chain 3: 100%|█████████▉| 1660/1667 [00:23<00:00, 100.40it/s]

Running chain 3: 100%|██████████| 1667/1667 [00:23<00:00, 70.77it/s] 

### 3.4 New data only (reference)
Full model from scratch on the new 10K, ignoring the base model.

$$\operatorname{logit} p_i = \alpha + x_i^\top \beta + v_i^\top \gamma, \qquad \alpha,\ \beta_j,\ \gamma_k \sim \mathcal{N}(0, 1)$$

$$p(\alpha, \beta, \gamma \mid D_\text{new}) \;\propto\; \prod_{i \in \text{new}} p_i^{y_i} (1 - p_i)^{1 - y_i} \;\cdot\; \mathcal{N}(\alpha; 0, 1) \prod_j \mathcal{N}(\beta_j; 0, 1) \prod_k \mathcal{N}(\gamma_k; 0, 1)$$

In [10]:
with pm.Model(coords={"feature": ALL_COLS}) as model_new:
    intercept = pm.Normal("intercept", mu=0, sigma=1)
    beta = pm.Normal("beta", mu=0, sigma=1, dims="feature")

    pm.Bernoulli("dead", logit_p=intercept + pm.math.dot(X.loc[in_new, ALL_COLS].values, beta),
                 observed=y_new)

    idata_new = pm.sample(667, tune=1000, chains=6, nuts_sampler="numpyro", random_seed=SEED)

NUTS[numpyro]: [intercept, beta]


  0%|          | 0/1667 [00:00<?, ?it/s]

Compiling.. :   0%|          | 0/1667 [00:00<?, ?it/s]

  0%|          | 0/1667 [00:00<?, ?it/s]

Compiling.. :   0%|          | 0/1667 [00:00<?, ?it/s]

...

Running chain 4: 100%|██████████| 1667/1667 [00:23<00:00, 70.25it/s]

Running chain 5: 100%|█████████▉| 1660/1667 [00:23<00:00, 87.08it/s]

Running chain 5: 100%|██████████| 1667/1667 [00:24<00:00, 69.44it/s]

Running chain 2: 100%|█████████▉| 1660/1667 [00:24<00:00, 94.88it/s]

Running chain 2: 100%|██████████| 1667/1667 [00:24<00:00, 68.64it/s]

### 3.5 Joint model with imputed vitals (reference, needs the old records)
All old + new patients in one full model. The old patients' vitals are unknown, so their risk is averaged over the vitals model.

$z_i$ = the three standardised vitals, $v(z_i) = (z_{i1}, z_{i1}^2, z_{i2}, z_{i2}^2, z_{i3}, z_{i3}^2)$, observed for new patients only.

$$\operatorname{logit} p_i = \alpha + x_i^\top \beta + v(z_i)^\top \gamma, \qquad z_i \sim \mathcal{N}(\nu_0 + B^\top x_i,\; L L^\top)$$

$$p(\alpha, \beta, \gamma, \nu_0, B, L \mid D_{\text{old}}, D_{\text{new}}) \;\propto\; \prod_{i \in \text{new}} p_i^{y_i} (1 - p_i)^{1 - y_i}\, \mathcal{N}(z_i;\, \nu_0 + B^\top x_i,\, L L^\top) \;\cdot\; \prod_{i \in \text{old}} \bar p_i^{\,y_i} (1 - \bar p_i)^{1 - y_i} \;\cdot\; \text{priors}$$

$$\bar p_i = \int \sigma\!\left(\alpha + x_i^\top \beta + v(z)^\top \gamma\right) \mathcal{N}(z;\, \nu_0 + B^\top x_i,\, L L^\top)\, dz \;\approx\; \sum_{k=1}^{125} w_k\, \sigma\!\left(\alpha + x_i^\top \beta + v(\nu_0 + B^\top x_i + L e_k)^\top \gamma\right)$$

Gauss–Hermite grid: 5 points per vital, 125 points $e_k$ with weights $w_k$. $\bar p_i$ depends only on $x_i$, so identical old patients are collapsed into one row with a count.

In [11]:
from numpy.polynomial.hermite_e import hermegauss

# an old patient's likelihood depends only on the base features (the vitals are averaged out),
# so identical old patients are collapsed into one row with a count, as in demo_adv
old = (X.loc[in_old, BASE_COLS].assign(dead=df.loc[in_old, "dead"])
         .groupby(BASE_COLS, as_index=False).agg(n=("dead", "size"), dead=("dead", "sum")))
Xb_old, n_old, d_old = old[BASE_COLS].values, old["n"].values, old["dead"].values
Xb_new = X.loc[in_new, BASE_COLS].values
Z_new = X.loc[in_new, VITALS].values                 # standardised vitals, observed for the new patients only

# Gauss–Hermite grid for averaging over a 3D standard normal: 5 points per vital, 125 points e_k with weights w_k
nodes, weights = hermegauss(5)
E = np.stack(np.meshgrid(nodes, nodes, nodes, indexing="ij"), axis=-1).reshape(-1, 3)
W = np.stack(np.meshgrid(weights, weights, weights, indexing="ij"), axis=-1).reshape(-1, 3).prod(axis=1)
log_w = np.log(W / W.sum())

with pm.Model(coords={"feature": ALL_COLS}) as model_joint:
    # vitals model, learned where the vitals are observed: z ~ N(nu0 + x·B, L·Lᵀ)
    nu0 = pm.Normal("nu0", mu=0, sigma=1, shape=3)
    B = pm.Normal("B", mu=0, sigma=1, shape=(len(BASE_COLS), 3))
    chol, _, _ = pm.LKJCholeskyCov("chol", n=3, eta=2, sd_dist=pm.Exponential.dist(1.0, shape=3))
    pm.MvNormal("z_new", mu=nu0 + pm.math.dot(Xb_new, B), chol=chol, observed=Z_new)

    # outcome model: the same full model for every patient
    intercept = pm.Normal("intercept", mu=0, sigma=1)
    beta = pm.Normal("beta", mu=0, sigma=1, dims="feature")

    # new patients: vitals observed
    pm.Bernoulli("dead_new", logit_p=intercept + pm.math.dot(X.loc[in_new, ALL_COLS].values, beta), observed=y_new)

    # old patients: average the risk over their unknown vitals, z = nu0 + x·B + L·e_k at every grid point
    z = (nu0 + pm.math.dot(Xb_old, B))[:, None, :] + pm.math.dot(E, chol.T)[None, :, :]       # rows × 125 × 3
    g = beta[len(BASE_COLS):]                                                                 # V1, V1², V2, V2², V3, V3²
    eta = ((intercept + pm.math.dot(Xb_old, beta[:len(BASE_COLS)]))[:, None]
           + g[0] * z[..., 0] + g[1] * z[..., 0] ** 2
           + g[2] * z[..., 1] + g[3] * z[..., 1] ** 2
           + g[4] * z[..., 2] + g[5] * z[..., 2] ** 2)
    log_p1 = pm.math.logsumexp(log_w - pm.math.log1pexp(-eta), axis=1, keepdims=False)    # log P(death | x)
    log_p0 = pm.math.logsumexp(log_w - pm.math.log1pexp(eta), axis=1, keepdims=False)     # log P(survival | x)
    pm.Potential("dead_old", pm.math.sum(d_old * log_p1 + (n_old - d_old) * log_p0))      # binomial, per row

    # dense mass matrix: lets NUTS learn the correlations between the vitals model and the outcome model
    idata_joint = pm.sample(667, tune=1000, chains=6, nuts_sampler="numpyro", random_seed=SEED,
                            nuts={"dense_mass": True})

NUTS[numpyro]: [nu0, B, chol, intercept, beta]


  0%|          | 0/1667 [00:00<?, ?it/s]

Compiling.. :   0%|          | 0/1667 [00:00<?, ?it/s]

  0%|          | 0/1667 [00:00<?, ?it/s]

Compiling.. :   0%|          | 0/1667 [00:00<?, ?it/s]

...

Running chain 3: 100%|██████████| 1667/1667 [06:32<00:00,  4.24it/s]

Running chain 1:  95%|█████████▍| 1577/1667 [06:32<00:11,  8.06it/s]

Running chain 1: 100%|█████████▉| 1660/1667 [06:34<00:00, 10.59it/s]

Running chain 1: 100%|██████████| 1667/1667 [06:35<00:00, 10.89it/s]

Running chain 1: 100%|██████████| 1667/1667 [06:35<00:00,  4.22it/s]

## 4. Evaluation
All models on the same 10K test patients.

### 4.1 Scores

In [12]:
X_test = X[in_test]
y_test = df.loc[in_test, "dead"].values
p_true_test = p_true[in_test].values

def predict(d):
    """Posterior mean P(death) per test patient; d = draws with an intercept column and one column per feature."""
    cols = list(d.columns[1:])
    Xt = X_test[cols].values
    a, b = d["intercept"].values, d[cols].values
    p_sum = np.zeros(len(Xt))
    for s in range(0, len(a), 250):                      # in chunks of draws to save memory
        p_sum += (1 / (1 + np.exp(-(a[s:s + 250, None] + b[s:s + 250] @ Xt.T)))).sum(axis=0)
    return p_sum / len(a)

def auc(y, p):
    """Probability that a random death gets a higher prediction than a random survivor."""
    r = pd.Series(p).rank().values
    n1, n0 = y.sum(), len(y) - y.sum()
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def scores(p):
    return {
        "AUC": auc(y_test, p),
        "Brier": np.mean((p - y_test) ** 2),
        "log loss": -np.mean(y_test * np.log(p) + (1 - y_test) * np.log(1 - p)),
        "mean |p - p_true|": np.abs(p - p_true_test).mean(),
    }

models = {
    "base (old data, no vitals)": base_draws,
    "fixed base + vitals": draws_of(idata_fixed, VITAL_COLS, base_lp=1.0),
    "recalibrated base + vitals": draws_of(idata_recal, RECAL_COLS),
    "old posterior as prior": draws_of(idata_prior, ALL_COLS),
    "new data only": draws_of(idata_new, ALL_COLS),
    "joint, imputed vitals": draws_of(idata_joint, ALL_COLS),
}
results = pd.DataFrame({name: scores(predict(d)) for name, d in models.items()}).T
results.loc["true probabilities"] = scores(p_true_test)
print(f"test patients: {len(y_test)}, deaths: {y_test.sum()}")
results.round(4)

test patients: 20000, deaths: 569


,AUC,Brier,log loss,mean |p - p_true|
"base (old data, no vitals)",0.7050,0.0271,0.1215,0.0122
fixed base + vitals,0.7509,0.0265,0.1167,0.0074
recalibrated base + vitals,0.7526,0.0264,0.1158,0.0039
old posterior as prior,0.7540,0.0264,0.1161,0.0053
new data only,0.7560,0.0264,0.1156,0.0031
"joint, imputed vitals",0.7555,0.0264,0.1156,0.0030
true probabilities,0.7593,0.0263,0.1152,0.0000
